In [13]:
# !pip install requests pandas sentence-transformers hdbscan google-generativeai jupyter

In [14]:
# !pip install streamlit requests sentence-transformers hdbscan pandas numpy google-genai

In [15]:
# %pip install umap-learn

In [16]:
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
import requests
import pandas as pd
import os
from google import genai
import numpy as np
from pydantic import BaseModel, Field
from google.genai.types import GenerateContentConfig

class TrendInsight(BaseModel):
    trend_name: str = Field(description="A catchy 2-to-4 word label for the trend.")
    key_ingredients_or_products: list[str] = Field(description="Specific products, ingredients, or tools explicitly mentioned in the posts.")
    consumer_pain_point: str = Field(description="The underlying problem or insecurity the consumers are trying to solve.")
    capitalization_strategy: str = Field(description="A 1-sentence idea on how a brand could capitalize on this specific trend. Make it directand actionable.")
    actionability_score: int = Field(description="A score from 1-10 on how easily a business could monetize this trend.")


# --- Credentials ---
BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [17]:
def fetch_bluesky_posts(query, target_count):

    # 1. Create a session to get the auth token
    session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    session_data = {"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD}
    session_resp = requests.post(session_url, json=session_data).json()
    
    if "accessJwt" not in session_resp:
        raise Exception(f"Failed to authenticate: {session_resp}")
        
    auth_token = session_resp["accessJwt"]
    headers = {"Authorization": f"Bearer {auth_token}"}
    
    # 2. Search for posts iteratively
    search_url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    
    posts_data = []
    cursor = None
    
    print(f"Fetching {target_count} posts for '{query}'...")
    while len(posts_data) < target_count:
        params = {"q": query, "limit": 100} # 100 is the max per request
        if cursor:
            params["cursor"] = cursor
            
        resp = requests.get(search_url, headers=headers, params=params).json()
        new_posts = resp.get("posts", [])
        
        if not new_posts:
            break # No more posts available
            
        for post in new_posts:
            # We extract the text, the timestamp, and the author
            posts_data.append({
                "text": post["record"]["text"],
                "created_at": post["record"]["createdAt"],
                "author": post["author"]["handle"],
                "replyCount": post.get("replyCount", 0),
                "repostCount": post.get("repostCount", 0),
                "likeCount": post.get("likeCount", 0),
                "quoteCount": post.get("quoteCount", 0)
                # "has_embed_link": has_embed,
                # "labels": labels
            })
            
        cursor = resp.get("cursor")
        if not cursor:
            break
            
    # Keep only the target amount and convert to a DataFrame
    df = pd.DataFrame(posts_data[:target_count])
    print(f"Successfully fetched {len(df)} posts.")
    return df

# Test
# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_posts.head()

In [18]:
def filter_spam_posts(df, threshold):
    """
    Calculates a spam score (0.0 to 1.0) based on engagement, duplication, 
    and text formatting, then filters out posts above the threshold.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df_scored = df.copy()
    
    # Initialize score
    df_scored['spam_score'] = 0.0
    
    # 1. Duplication Penalty (Strongest signal)
    # Flag posts that share the exact same text (e.g., cross-posting bots)
    is_duplicate = df_scored.duplicated(subset=['text'], keep='first')
    df_scored.loc[is_duplicate, 'spam_score'] += 0.5
    
    # 2. Low Engagement Penalty
    # Summing up the engagement metrics you are already fetching
    df_scored['total_engagement'] = (
        df_scored['replyCount'] + 
        df_scored['repostCount'] + 
        df_scored['likeCount'] + 
        df_scored['quoteCount']
    )
    # Add a penalty if the post has completely zero engagement
    df_scored.loc[df_scored['total_engagement'] == 0, 'spam_score'] += 0.2
    
    # 3. Content Heuristics (Links & Hashtags)
    # Count occurrences using basic regex
    df_scored['hashtag_count'] = df_scored['text'].str.count(r'#\w+')
    df_scored['link_count'] = df_scored['text'].str.count(r'http[s]?://')
    
    # Penalize spammy text formatting
    df_scored.loc[df_scored['hashtag_count'] > 4, 'spam_score'] += 0.15
    df_scored.loc[df_scored['link_count'] >= 2, 'spam_score'] += 0.15
    
    # 4. Cap the maximum score at 1.0
    df_scored['spam_score'] = df_scored['spam_score'].clip(upper=1.0)
    
    # Filter the DataFrame based on the acceptable threshold
    initial_count = len(df_scored)
    df_filtered = df_scored[df_scored['spam_score'] < threshold].copy()
    filtered_count = len(df_filtered)
    
    print(f"Filtered out {initial_count - filtered_count} spam-likely posts.")
    
    # Clean up calculation columns before passing to the clustering phase
    df_filtered = df_filtered.drop(columns=['total_engagement', 'hashtag_count', 'link_count'])
    
    return df_filtered


# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_clean = filter_spam_posts(df_posts, threshold=0.6)
# df_clustered = cluster_social_posts(df_clean)

In [19]:

df_posts = fetch_bluesky_posts("toothpaste", target_count=5000)
df_clean = filter_spam_posts(df_posts, threshold=0.1)

df_posts



Fetching 5000 posts for 'toothpaste'...
Successfully fetched 986 posts.
Filtered out 296 spam-likely posts.


,text,created_at,author,replyCount,repostCount,likeCount,quoteCount
0,WHO SAYS MINT CHOC CHIP?????\n\nThat's like ea...,2026-07-31T21:08:50.813Z,deaconbiker.bsky.social,0,0,0,0
1,Toothbrush and toothpaste should be used after...,2026-07-31T20:57:42.295Z,mecha-sheep.bsky.social,0,0,1,0
2,What about other food / medicines? Is toothpas...,2026-07-31T20:43:51.981Z,melodyschreiber.com,1,0,0,0
3,Toothpaste probably! *Keeps pushing*,2026-07-31T20:11:26.315Z,hmmmbear.bsky.social,1,0,1,0
4,If you did- would it be like a cat kneading do...,2026-07-31T20:10:15.199Z,audrascollective.bsky.social,1,0,0,0
...,...,...,...,...,...,...,...
981,"When they are finished saving the world, how a...",2026-07-21T16:21:46.943Z,retrobob.bsky.social,1,0,0,0
982,"Etee Canada has some wonderful home products, ...",2026-07-21T16:14:48.654Z,minibubbly.bsky.social,1,6,12,2
983,tbh I was thinking about that with the prophec...,2026-07-21T16:07:34.564Z,ddder.bsky.social,0,0,1,0
984,Do this do this do this. Also donate things li...,2026-07-21T15:21:17.951Z,kaitstrong.bsky.social,2,0,1,0


In [20]:
# df_clean

In [ ]:
def cluster_social_posts(df, cluster_fraction, sample_fraction):
    print("Loading Sentence Transformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2') 
    
    print("Generating embeddings...")
    embeddings = model.encode(df['text'].tolist())
    
    print("Reducing dimensions with UMAP...")
    # Compress the 384 dimensions down to 5 to help HDBSCAN find density
    umap_model = umap.UMAP(
        n_neighbors=30, # Focuses on local neighborhood size 
        n_components=5, # Reduce to 5 dimensions
        min_dist=0.0,   # How tightly to pack points together 
        metric='cosine',# Cosine works best for text embeddings
        random_state=42 # Ensure reproducible results
    )
    reduced_embeddings = umap_model.fit_transform(embeddings)
    
    print("Running HDBSCAN clustering...")
    
    # Calculate dynamic parameters based on DataFrame size
    total_posts = len(df)
    
    # Force the values to be integers, and set an absolute minimum floor (e.g., 5)
    # so small datasets don't end up with a min_cluster_size of 1.
    dynamic_min_cluster_size = max(5, int(total_posts * cluster_fraction))
    dynamic_min_samples = max(5, int(dynamic_min_cluster_size * 0.5))
    
    print(f"Dynamic Settings: min_cluster_size={dynamic_min_cluster_size}, min_samples={dynamic_min_samples}")
    
    print("Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=dynamic_min_cluster_size, 
        min_samples=dynamic_min_samples,      
        metric='euclidean'
        # , cluster_selection_epsilon=0.05  
    )


    df['cluster_id'] = clusterer.fit_predict(reduced_embeddings)
    
    # -1 means "noise" (unclustered). Let's filter those out.
    clustered_df = df[df['cluster_id'] != -1]
    
    print(f"Found {len(clustered_df['cluster_id'].unique())} unique clusters.")
    return clustered_df
# Test
# df_clustered = cluster_social_posts(df_clean, 0.01 , 0.002)
# print(df_clustered['cluster_id'].value_counts())

In [22]:
# Test keywords

lst_keywords = ["skincare", "toothpaste", "haircare", "electric vehicle", "mens fashion", "stocks", "investment", "fitness"]

for keyword in lst_keywords:
    print(keyword)
    df_clustered = cluster_social_posts(df_clean, 0.01 , 0.002)
    print(df_clustered['cluster_id'].value_counts())
    df_clustered = pd.DataFrame()



skincare
Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...
Reducing dimensions with UMAP...


c:\Users\JesusSantillanMinila\Documents\GitHub\social-media-trend-detector\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running HDBSCAN clustering...
Dynamic Settings: min_cluster_size=6, min_samples=5
Running HDBSCAN clustering...
Found 16 unique clusters.
cluster_id
9     80
0     73
3     67
12    51
13    50
11    29
14    26
6     21
10    14
15    11
4     10
1     10
8      9
7      7
2      6
5      6
Name: count, dtype: int64
toothpaste
Loading Sentence Transformer model...


...

Running HDBSCAN clustering...
Dynamic Settings: min_cluster_size=6, min_samples=5
Running HDBSCAN clustering...
Found 16 unique clusters.
cluster_id
9     80
0     73
3     67
12    51
13    50
11    29
14    26
6     21
10    14
15    11
4     10
1     10
8      9
7      7
2      6
5      6
Name: count, dtype: int64
fitness
Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...
Reducing dimensions with UMAP...


c:\Users\JesusSantillanMinila\Documents\GitHub\social-media-trend-detector\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running HDBSCAN clustering...
Dynamic Settings: min_cluster_size=6, min_samples=5
Running HDBSCAN clustering...
Found 16 unique clusters.
cluster_id
9     80
0     73
3     67
12    51
13    50
11    29
14    26
6     21
10    14
15    11
4     10
1     10
8      9
7      7
2      6
5      6
Name: count, dtype: int64


In [23]:
total_posts = len(df_clean)
total_posts

690

In [24]:
def extract_actionable_insights(df_clustered):
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    # Create a dictionary to hold our rich insights
    cluster_insights = {}
    # Count the posts in each cluster and sort them from largest to smallest
    cluster_sizes = df_clustered['cluster_id'].value_counts()
    
    # Grab the IDs of the top 6 largest clusters
    top_6_clusters = cluster_sizes.head(6).index.tolist()
    
    # Iterate ONLY over those top 6
    for cluster_id in top_6_clusters:
        # Increase the sample size slightly for better context
        sample_posts = df_clustered[df_clustered['cluster_id'] == cluster_id]['text'].head(15).tolist()

        posts_text = "\n- ".join(sample_posts)
        
        prompt = f"""
        You are an expert consumer trend analyst and product developer. 
        Analyze the following social media posts that have been clustered together:
        - {posts_text}
        
        Extract the underlying trend and identify exactly how a business can capitalize on it.
        """
        
        # Enforce structured output via GenerateContentConfig
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt,
            config=GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=TrendInsight,
            )
        )
        
        # Access the structured data safely using .parsed
        insight = response.parsed
        cluster_insights[cluster_id] = insight
        
        print(f"Analyzed Cluster {cluster_id}: {insight.trend_name}. \nProduct: {insight.capitalization_strategy} (Score: {insight.actionability_score}/10)")
        
    # Map the new structured data back to the dataframe
    df_clustered['trend_name'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].trend_name if x in cluster_insights else None)
    df_clustered['key_products'] = df_clustered['cluster_id'].map(lambda x: ", ".join(cluster_insights[x].key_ingredients_or_products) if x in cluster_insights else None)
    df_clustered['pain_point'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].consumer_pain_point if x in cluster_insights else None)
    df_clustered['strategy'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].capitalization_strategy if x in cluster_insights else None)
    df_clustered['actionability_score'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].actionability_score if x in cluster_insights else None)
    
    return df_clustered

In [25]:

# Test. Keep commented until ready to run gemini API
# df_labeled = extract_actionable_insights(df_clustered)